# Tuần 2 (Chỉnh Sửa Toàn Diện): Thực Nghiệm Chuyển Giao Tri Thức (Pretrained Models) Trên GPU
### Đề tài: Hệ Thống Xác Thực Chữ Ký Ngân Hàng Tự Động (AIO2026 — Module 05)
**Học viên thực hiện:** Trần Phương Bình  
**Phiên bản:** Cập nhật tiếp thu 100% góp ý chuyên môn từ Trợ giảng (TA)

---

## 🎯 1. Mục Tiêu Thực Nghiệm Định Lượng
Notebook này được thiết kế để chạy thực nghiệm đo kiểm hiệu năng thực tế trên **Google Colab (GPU T4 / CPU)**:
1. **Khắc phục lỗi `pretrained=False`:** Kích hoạt `pretrained=True` tải chính thức trọng số ImageNet-1k để kiểm chứng bản chất chuyển giao tri thức.
2. **Khớp chính xác số lượng tham số:** Xác minh tự động số tham số Trainable giữa code `src/models.py` và báo cáo (Linear Probing $2,049$ và $769$ params).
3. **Bổ sung cấu hình đối chứng Siamese Network:** So sánh trực diện kiến trúc kinh điển **Siamese ResNet-50** với giải pháp **3-Channel Difference Stacking**.
4. **Giải trình toán học Focal Loss:** Khảo sát cơ chế Hard-Sample Mining trên bộ dữ liệu cặp đối chiếu cân bằng 50/50.
5. **Thu thập số liệu định lượng (Empirical Benchmark):** Đo lường Val Loss, Val Accuracy, ROC-AUC, EER và xuất biểu đồ **Learning Curves** làm minh chứng học thuật vững chắc.


In [ ]:
# 1. Cấu hình môi trường tự động (Tự động Clone Repo trên Google Colab)
import os
import sys
import shutil
import subprocess
from pathlib import Path

# Kiểm tra nếu đang chạy trên Google Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[*] Đang chạy trên Google Colab. Bắt đầu cấu hình tự động...")
    REPO_NAME = "Automatic-Signature-Verification-System-ASVS-"
    REPO_URL = f"https://github.com/AIVIETNAM-AIO-TPBINH/{REPO_NAME}.git"
    
    # Clone hoặc pull mới nhất nếu đã có
    if not os.path.exists(f"/content/{REPO_NAME}"):
        print(f"[*] Đang clone mã nguồn từ {REPO_URL}...")
        !git clone {REPO_URL} /content/{REPO_NAME}
    else:
        print("[*] Repo đã tồn tại, đang cập nhật mã nguồn mới nhất...")
        !cd /content/{REPO_NAME} && git pull origin main
    
    # Chuyển thư mục làm việc vào thư mục repo
    os.chdir(f"/content/{REPO_NAME}")
    
    # Cài đặt các thư viện cần thiết
    !pip install -q timm scikit-learn seaborn matplotlib tqdm

# Thiết lập đường dẫn gốc dự án vào sys.path
CURRENT_DIR = Path().resolve()
if (CURRENT_DIR / "src").exists():
    BASE_DIR = CURRENT_DIR
elif Path(f"/content/Automatic-Signature-Verification-System-ASVS-/src").exists():
    BASE_DIR = Path(f"/content/Automatic-Signature-Verification-System-ASVS-")
    os.chdir(BASE_DIR)
else:
    BASE_DIR = CURRENT_DIR

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

print(f"[+] Thư mục làm việc hiện tại: {os.getcwd()}")

# Import các module từ src/
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import set_seed, get_device
from src.dataset import get_dataloaders
from src.models import build_model
from src.losses import get_loss_function
from src.trainer import SignatureTrainer

set_seed(42)
device = get_device()
print(f"[+] Thiết bị tính toán: {device}")
if torch.cuda.is_available():
    print(f"[+] GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"[+] VRAM khả dụng: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## 📦 2. Kiểm Tra & Tự Động Định Vị Ảnh CEDAR
Tự động tìm kiếm file `data.zip`, giải nén và sắp xếp thư mục `data/raw/signatures/full_org` và `full_forg` hoàn toàn tự động!


In [ ]:
# Kiểm tra và tự động giải nén / sắp xếp thư mục ảnh
import zipfile
import shutil

org_dir = Path("data/raw/signatures/full_org")
forg_dir = Path("data/raw/signatures/full_forg")

def has_images(dir_path):
    return dir_path.exists() and len(list(dir_path.glob("*.png"))) > 0

# Tự động di chuyển nếu signatures đang nằm ở vị trí khác
if not (has_images(org_dir) and has_images(forg_dir)):
    # Các vị trí signatures có thể đang tồn tại
    alt_locations = [
        Path("signatures"),
        Path("/content/signatures"),
        Path("../signatures"),
        Path("/content/Automatic-Signature-Verification-System-ASVS-/signatures")
    ]
    for alt in alt_locations:
        if (alt / "full_org").exists():
            print(f"[*] Phát hiện ảnh tại '{alt}'. Đang tự động chuyển vào 'data/raw/signatures/'...")
            os.makedirs("data/raw", exist_ok=True)
            if Path("data/raw/signatures").exists():
                shutil.rmtree("data/raw/signatures")
            shutil.copytree(str(alt), "data/raw/signatures")
            break

# Nếu vẫn chưa có, tìm data.zip để giải nén
if not (has_images(org_dir) and has_images(forg_dir)):
    zip_candidates = [
        Path("/content/data.zip"),
        Path("data.zip"),
        Path("/content/drive/MyDrive/data.zip"),
        Path("../data.zip")
    ]
    found_zip = None
    for cand in zip_candidates:
        if cand.exists():
            found_zip = cand
            break
            
    if found_zip:
        print(f"[*] Tìm thấy {found_zip}. Đang giải nén trực tiếp vào data/raw/...")
        os.makedirs("data/raw", exist_ok=True)
        with zipfile.ZipFile(found_zip, 'r') as zip_ref:
            # File data.zip chứa sẵn folder 'signatures/' -> giải nén vào data/raw cho ra data/raw/signatures/
            zip_ref.extractall("data/raw")
        print("[+] Đã giải nén xong data.zip!")
    else:
        print("[!] Không tìm thấy data.zip. Hãy đảm bảo file data.zip đã được tải lên /content/data.zip!")

num_org = len(list(org_dir.glob("*.png"))) if org_dir.exists() else 0
num_forg = len(list(forg_dir.glob("*.png"))) if forg_dir.exists() else 0
print(f"[*] Kết quả kiểm tra ảnh:")
print(f"    - Số ảnh gốc (full_org): {num_org} / 1,320")
print(f"    - Số ảnh giả mạo (full_forg): {num_forg} / 1,320")

# Khởi tạo DataLoader
batch_size = 64 if torch.cuda.is_available() else 32
train_loader, val_loader, test_loader = get_dataloaders(
    processed_dir="data/processed",
    batch_size=batch_size,
    num_workers=2 if torch.cuda.is_available() else 0
)

print(f"[*] Dữ liệu sẵn sàng:")
print(f"    - Số cặp Train: {len(train_loader.dataset):,} | Số batch: {len(train_loader)}")
print(f"    - Số cặp Val:   {len(val_loader.dataset):,} | Số batch: {len(val_loader)}")
print(f"    - Số cặp Test:  {len(test_loader.dataset):,} | Số batch: {len(test_loader)}")


## 🏗️ 3. Kiểm Định Tự Động Ma Trận Tham Số 8 Mô Hình (Khớp 100% Code & Báo Cáo)


In [ ]:
# Ma trận 8 cấu hình thực nghiệm đối chứng
configs_to_inspect = [
    ("mlp_baseline", "1. MLP Baseline (Duỗi phẳng 2D)"),
    ("custom_cnn", "2. Custom 2D-CNN (4 Blocks Scratch)"),
    ("resnet50_feature_extraction", "3. ResNet-50 Feature Extraction (Linear Head 2,049 params)"),
    ("resnet50_partial_freezing", "4. ResNet-50 Partial Freezing (Stage 3-4 FT)"),
    ("resnet50_discriminative_finetuning", "5. ResNet-50 Discriminative FT (Ứng viên Production 1)"),
    ("vit_base_feature_extraction", "6. ViT-Base Feature Extraction (Linear Head 769 params)"),
    ("vit_base_full_finetuning", "7. ViT-Base Full Fine-tuning (Ứng viên Production 2)"),
    ("siamese_resnet50", "8. Siamese ResNet-50 (Cấu hình đối chứng kinh điển)"),
]

table_rows = []
for key, label in configs_to_inspect:
    m = build_model(key, pretrained=False)
    total_p = sum(p.numel() for p in m.parameters())
    trainable_p = sum(p.numel() for p in m.parameters() if p.requires_grad)
    frozen_p = total_p - trainable_p
    
    table_rows.append({
        "STT": label.split(".")[0],
        "Cấu hình thử nghiệm": label.split(".")[1].strip(),
        "Tổng tham số": f"{total_p:,}",
        "Tham số Trainable": f"{trainable_p:,}",
        "Tham số Frozen": f"{frozen_p:,}",
        "Tỷ lệ Trainable": f"{(trainable_p / total_p)*100:.3f}%"
    })

df_params = pd.DataFrame(table_rows)
print(df_params.to_string(index=False))


## ⚖️ 4. Cơ Chế Toán Học: Focal Loss Đóng Vai Trò Bộ Khai Thác Mẫu Khó (Hard-Sample Miner)
$$\mathcal{L}_{\text{FL}}(p_t) = -(1 - p_t)^\gamma \log(p_t) \quad (\gamma = 2.0)$$
- Mẫu dễ ($p_t = 0.99$): $(1 - 0.99)^2 = 0.0001 \to$ Triệt tiêu $10,000$ lần gradient!
- Mẫu khó ($p_t = 0.50$): $(1 - 0.50)^2 = 0.2500 \to$ Giữ nguyên gradient tối đa để ép mạng học nét ký tinh vi!


## 🚀 5. Huấn Luyện Thực Nghiệm Đối Chứng Các Cấu Hình Chủ Đạo (Pretrained=True)


In [ ]:
# Số epoch điều chỉnh linh hoạt theo phần cứng:
# Nếu có GPU: chạy 5 epoch mỗi mô hình.
# Nếu chạy CPU: chạy 2 epoch cho các mô hình Feature Extraction / Discriminative để có số liệu nhanh.
num_epochs = 5 if torch.cuda.is_available() else 2

benchmark_models = {
    "resnet50_linear_probing": {
        "name": "ResNet-50 (Feature Extraction)",
        "model_key": "resnet50_feature_extraction",
        "lr_type": "head_only",
        "lr": 1e-3,
        "loss": "bce",
        "epochs": num_epochs
    },
    "resnet50_discriminative": {
        "name": "ResNet-50 (Discriminative FT)",
        "model_key": "resnet50_discriminative_finetuning",
        "lr_type": "discriminative",
        "loss": "focal",
        "epochs": num_epochs
    },
    "siamese_resnet50": {
        "name": "Siamese ResNet-50 (Shared Backbone)",
        "model_key": "siamese_resnet50",
        "lr_type": "discriminative",
        "loss": "focal",
        "epochs": num_epochs
    }
}

# Nếu có GPU, thêm ViT-Base vào benchmark
if torch.cuda.is_available():
    benchmark_models["vit_base_finetuning"] = {
        "name": "ViT-Base (Full Fine-tuning)",
        "model_key": "vit_base_full_finetuning",
        "lr_type": "vit_lr",
        "loss": "focal",
        "epochs": 5
    }

experiment_results = {}

for exp_id, cfg in benchmark_models.items():
    print(f"\n{'='*70}")
    print(f"[*] BẮT ĐẦU HUẤN LUYỆN: {cfg['name']}")
    print(f"{'='*70}")
    
    # Khởi tạo mô hình với trọng số Pretrained chính thức
    model = build_model(cfg["model_key"], pretrained=True)
    
    if cfg["lr_type"] == "head_only":
        optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=cfg["lr"],
            weight_decay=1e-2
        )
    elif cfg["lr_type"] == "discriminative":
        param_groups = model.get_parameter_groups(lr_early=1e-5, lr_mid=5e-5, lr_head=1e-3, weight_decay=1e-2)
        optimizer = torch.optim.AdamW(param_groups)
    elif cfg["lr_type"] == "vit_lr":
        param_groups = model.get_parameter_groups(lr_backbone=1e-5, lr_head=1e-4, weight_decay=1e-2)
        optimizer = torch.optim.AdamW(param_groups)
        
    loss_fn = get_loss_function(cfg["loss"])
    
    trainer = SignatureTrainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        loss_fn=loss_fn,
        device=str(device),
        model_name=exp_id,
        use_amp=True
    )
    
    history = trainer.fit(epochs=cfg["epochs"], early_stopping_patience=3)
    final_val_metrics = trainer.evaluate(val_loader)
    
    experiment_results[exp_id] = {
        "name": cfg["name"],
        "history": history,
        "final_metrics": final_val_metrics
    }

os.makedirs("reports", exist_ok=True)
with open("reports/training_benchmark_history.json", "w", encoding="utf-8") as f:
    json.dump({k: {"name": v["name"], "metrics": v["final_metrics"]} for k, v in experiment_results.items()}, f, indent=2)
print("\n[+] Đã lưu toàn bộ lịch sử huấn luyện vào reports/training_benchmark_history.json!")


## 📊 6. Bảng Đo Lường Hiệu Năng Thực Nghiệm & Biểu Đồ Learning Curves


In [ ]:
# Bảng tổng hợp số liệu định lượng
results_summary = []
for exp_id, data in experiment_results.items():
    m = data["final_metrics"]
    results_summary.append({
        "Mô Hình": data["name"],
        "Val Loss": f"{m.get('loss', 0.0):.4f}",
        "Val Accuracy (%)": f"{m.get('accuracy', 0.0)*100:.2f}%",
        "Val ROC-AUC": f"{m.get('roc_auc', 0.5):.4f}",
        "Val EER (%)": f"{m.get('eer', 0.0)*100:.2f}%"
    })

df_res = pd.DataFrame(results_summary)
print(df_res.to_string(index=False))

# Biểu đồ Learning Curves
num_models = len(experiment_results)
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Subplot 1: Train Loss
ax = axes[0, 0]
for exp_id, data in experiment_results.items():
    ax.plot(data["history"]["train_loss"], label=data["name"], marker='o')
ax.set_title("Training Loss qua các Epoch", fontsize=12, fontweight='bold')
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()

# Subplot 2: Val Loss
ax = axes[0, 1]
for exp_id, data in experiment_results.items():
    ax.plot(data["history"]["val_loss"], label=data["name"], marker='s')
ax.set_title("Validation Loss qua các Epoch", fontsize=12, fontweight='bold')
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()

# Subplot 3: Val Accuracy
ax = axes[1, 0]
for exp_id, data in experiment_results.items():
    ax.plot([acc * 100 for acc in data["history"]["val_acc"]], label=data["name"], marker='^')
ax.set_title("Validation Accuracy (%) qua các Epoch", fontsize=12, fontweight='bold')
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy (%)")
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()

# Subplot 4: Val EER
ax = axes[1, 1]
for exp_id, data in experiment_results.items():
    ax.plot([eer * 100 for eer in data["history"]["val_eer"]], label=data["name"], marker='d')
ax.set_title("Validation Equal Error Rate (EER %) (Càng thấp càng tốt)", fontsize=12, fontweight='bold')
ax.set_xlabel("Epoch")
ax.set_ylabel("EER (%)")
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()

plt.tight_layout()
os.makedirs("reports/figures", exist_ok=True)
plt.savefig("reports/figures/02_transfer_learning_curves.png", dpi=300)
plt.show()
print("[+] Đã lưu đồ thị Learning Curves vào reports/figures/02_transfer_learning_curves.png!")


## 📝 7. Nhận Định Kỹ Thuật Rút Ra Dựa Trên Số Liệu Thực Chứng
